In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [2]:
# Load the three datasets
disasters = pd.read_csv('disastersfinal.csv')
gdp = pd.read_csv('GDPfinal.csv')
population = pd.read_csv('Popfinal.csv')

# Quick sanity check — print the shape and first few rows of each
print("DISASTERS")
print(f"Shape: {disasters.shape}")
print(disasters.head())
print()

print("GDP")
print(f"Shape: {gdp.shape}")
print(gdp.head())
print()

print("POPULATION")
print(f"Shape: {population.shape}")
print(population.head())

DISASTERS
Shape: (403, 7)
                                                Name          Disaster  \
0   Southern Severe Storms and Flooding (April 1980)          Flooding   
1                      Hurricane Allen (August 1980)  Tropical Cyclone   
2  Central/Eastern Drought/Heat Wave (Summer-Fall...           Drought   
3                      Florida Freeze (January 1981)            Freeze   
4  Severe Storms, Flash Floods, Hail, Tornadoes (...      Severe Storm   

   Begin Date  End Date  CPI-Adjusted Cost  Unadjusted Cost  Deaths  
0    19800410  19800417             2749.4            706.8       7  
1    19800807  19800811             2236.2            590.0      13  
2    19800601  19801130            40681.2          10020.0    1260  
3    19810112  19810114             2076.4            572.0       0  
4    19810505  19810510             1409.1            401.4      20  

GDP
Shape: (45, 2)
  observation_date       GDP
0         1/1/1980  7257.316
1         1/1/1981  7441.485
2 

In [3]:
# Clean disasters
disasters.columns = ['name', 'disaster_type', 'begin_date', 'end_date',
                     'cpi_adjusted_cost', 'unadjusted_cost', 'deaths']

# Convert YYYYMMDD integer dates to datetime
disasters['begin_date'] = pd.to_datetime(disasters['begin_date'], format='%Y%m%d')
disasters['end_date'] = pd.to_datetime(disasters['end_date'], format='%Y%m%d')

# Extract year for joining with annual datasets
disasters['year'] = disasters['begin_date'].dt.year

# Clean GDP and population — rename columns and extract year
gdp = gdp.rename(columns={'observation_date': 'date', 'GDP': 'gdp_billions'})
gdp['date'] = pd.to_datetime(gdp['date'])
gdp['year'] = gdp['date'].dt.year

population = population.rename(columns={'observation_date': 'date', 'population': 'population'})
population['date'] = pd.to_datetime(population['date'])
population['year'] = population['date'].dt.year

# Verify the cleaning worked
print("DISASTERS (cleaned)")
print(disasters.head())
print(disasters.dtypes)
print()

print("GDP (cleaned)")
print(gdp.head())
print()

print("POPULATION (cleaned)")
print(population.head())

DISASTERS (cleaned)
                                                name     disaster_type  \
0   Southern Severe Storms and Flooding (April 1980)          Flooding   
1                      Hurricane Allen (August 1980)  Tropical Cyclone   
2  Central/Eastern Drought/Heat Wave (Summer-Fall...           Drought   
3                      Florida Freeze (January 1981)            Freeze   
4  Severe Storms, Flash Floods, Hail, Tornadoes (...      Severe Storm   

  begin_date   end_date  cpi_adjusted_cost  unadjusted_cost  deaths  year  
0 1980-04-10 1980-04-17             2749.4            706.8       7  1980  
1 1980-08-07 1980-08-11             2236.2            590.0      13  1980  
2 1980-06-01 1980-11-30            40681.2          10020.0    1260  1980  
3 1981-01-12 1981-01-14             2076.4            572.0       0  1981  
4 1981-05-05 1981-05-10             1409.1            401.4      20  1981  
name                         object
disaster_type                object
begin_d

In [4]:
# Aggregate disasters to yearly totals
yearly_disasters = disasters.groupby('year').agg(
    event_count=('name', 'count'),
    total_cost_millions=('cpi_adjusted_cost', 'sum'),
    total_deaths=('deaths', 'sum'),
    avg_cost_per_event=('cpi_adjusted_cost', 'mean'),
    max_cost=('cpi_adjusted_cost', 'max')
).reset_index()

print(yearly_disasters.head(10))
print(f"\nShape: {yearly_disasters.shape}")
print(f"Year range: {yearly_disasters['year'].min()} - {yearly_disasters['year'].max()}")

   year  event_count  total_cost_millions  total_deaths  avg_cost_per_event  \
0  1980            3              45666.8          1280        15222.266667   
1  1981            2               3485.5            20         1742.750000   
2  1982            5              15175.7           243         3035.140000   
3  1983            4              26755.5           186         6688.875000   
4  1984            2               3208.0            81         1604.000000   
5  1985            7              22514.6           379         3216.371429   
6  1986            3               7869.6           123         2623.200000   
7  1988            1              54610.4           454        54610.400000   
8  1989            6              40330.6           228         6721.766667   
9  1990            4              14589.4            30         3647.350000   

   max_cost  
0   40681.2  
1    2076.4  
2    4946.2  
3    9574.0  
4    1851.6  
5    4340.4  
6    5194.5  
7   54610.4  
8   

In [5]:
# Create a full range of years from 1980 to 2024
all_years = pd.DataFrame({'year': range(1980, 2025)})

# Merge with yearly_disasters, filling missing years with zeros
yearly_disasters = all_years.merge(yearly_disasters, on='year', how='left')

# Replace NaN values with 0 for the aggregated metrics
yearly_disasters = yearly_disasters.fillna(0)

# Cast counts and deaths to integer since they should be whole numbers
yearly_disasters['event_count'] = yearly_disasters['event_count'].astype(int)
yearly_disasters['total_deaths'] = yearly_disasters['total_deaths'].astype(int)

# Verify
print(yearly_disasters)
print(f"\nShape: {yearly_disasters.shape}")
print(f"Year range: {yearly_disasters['year'].min()} - {yearly_disasters['year'].max()}")

# Sanity check — which years have zero events?
zero_event_years = yearly_disasters[yearly_disasters['event_count'] == 0]['year'].tolist()
print(f"\nYears with zero billion-dollar disasters: {zero_event_years}")

    year  event_count  total_cost_millions  total_deaths  avg_cost_per_event  \
0   1980            3              45666.8          1280        15222.266667   
1   1981            2               3485.5            20         1742.750000   
2   1982            5              15175.7           243         3035.140000   
3   1983            4              26755.5           186         6688.875000   
4   1984            2               3208.0            81         1604.000000   
5   1985            7              22514.6           379         3216.371429   
6   1986            3               7869.6           123         2623.200000   
7   1987            0                  0.0             0            0.000000   
8   1988            1              54610.4           454        54610.400000   
9   1989            6              40330.6           228         6721.766667   
10  1990            4              14589.4            30         3647.350000   
11  1991            4              19611

In [6]:
# Join GDP and population to yearly_disasters
analysis = yearly_disasters.merge(gdp[['year', 'gdp_billions']], on='year', how='left')
analysis = analysis.merge(population[['year', 'population']], on='year', how='left')

# Quick check
print(analysis.head())
print(f"\nShape: {analysis.shape}")
print(f"\nAny missing values?")
print(analysis.isnull().sum())

   year  event_count  total_cost_millions  total_deaths  avg_cost_per_event  \
0  1980            3              45666.8          1280        15222.266667   
1  1981            2               3485.5            20         1742.750000   
2  1982            5              15175.7           243         3035.140000   
3  1983            4              26755.5           186         6688.875000   
4  1984            2               3208.0            81         1604.000000   

   max_cost  gdp_billions  population  
0   40681.2      7257.316   227225000  
1    2076.4      7441.485   229466000  
2    4946.2      7307.314   231664000  
3    9574.0      7642.266   233792000  
4    1851.6      8195.295   235825000  

Shape: (45, 8)

Any missing values?
year                   0
event_count            0
total_cost_millions    0
total_deaths           0
avg_cost_per_event     0
max_cost               0
gdp_billions           0
population             0
dtype: int64


In [7]:
# Derived metrics — the analytical core

# 1. Cost as percentage of GDP (gdp is in billions, cost is in millions, so divide cost by 1000 to get billions)
analysis['cost_pct_of_gdp'] = (analysis['total_cost_millions'] / 1000) / analysis['gdp_billions'] * 100

# 2. Events per million people
analysis['events_per_million'] = analysis['event_count'] / (analysis['population'] / 1_000_000)

# 3. Cost per capita (in dollars, not millions)
analysis['cost_per_capita_dollars'] = (analysis['total_cost_millions'] * 1_000_000) / analysis['population']

# 4. Deaths per million people
analysis['deaths_per_million'] = analysis['total_deaths'] / (analysis['population'] / 1_000_000)

# 5. Decade (for decade-level comparisons)
analysis['decade'] = (analysis['year'] // 10) * 10

# Verify
print(analysis.head())
print(f"\nColumn list:")
print(analysis.columns.tolist())
print(f"\nShape: {analysis.shape}")

   year  event_count  total_cost_millions  total_deaths  avg_cost_per_event  \
0  1980            3              45666.8          1280        15222.266667   
1  1981            2               3485.5            20         1742.750000   
2  1982            5              15175.7           243         3035.140000   
3  1983            4              26755.5           186         6688.875000   
4  1984            2               3208.0            81         1604.000000   

   max_cost  gdp_billions  population  cost_pct_of_gdp  events_per_million  \
0   40681.2      7257.316   227225000         0.629252            0.013203   
1    2076.4      7441.485   229466000         0.046839            0.008716   
2    4946.2      7307.314   231664000         0.207678            0.021583   
3    9574.0      7642.266   233792000         0.350099            0.017109   
4    1851.6      8195.295   235825000         0.039144            0.008481   

   cost_per_capita_dollars  deaths_per_million  decade  

In [8]:
# Statistical trend analysis
from scipy.stats import linregress

# Install pymannkendall if not already installed
!pip install pymannkendall -q
import pymannkendall as mk

def analyze_trend(series, label):
    """Run both Mann-Kendall and linear regression on a time series"""
    # Drop any NaN values
    clean = series.dropna()

    # Mann-Kendall trend test
    mk_result = mk.original_test(clean)

    # Linear regression vs. year
    years = analysis.loc[clean.index, 'year']
    reg = linregress(years, clean)

    print(f"--- {label} ---")
    print(f"Mann-Kendall: trend={mk_result.trend}, p={mk_result.p:.4f}, Sen's slope={mk_result.slope:.4f}")
    print(f"Linear Regression: slope={reg.slope:.4f}, R²={reg.rvalue**2:.4f}, p={reg.pvalue:.4f}")
    print()

# Run on key metrics
analyze_trend(analysis['event_count'], 'Event Count')
analyze_trend(analysis['total_cost_millions'], 'Total Cost (millions)')
analyze_trend(analysis['cost_pct_of_gdp'], 'Cost as % of GDP')
analyze_trend(analysis['events_per_million'], 'Events per Million People')
analyze_trend(analysis['total_deaths'], 'Total Deaths')

--- Event Count ---
Mann-Kendall: trend=increasing, p=0.0000, Sen's slope=0.3750
Linear Regression: slope=0.4262, R²=0.7028, p=0.0000

--- Total Cost (millions) ---
Mann-Kendall: trend=increasing, p=0.0000, Sen's slope=1804.5966
Linear Regression: slope=3133.3739, R²=0.2842, p=0.0002

--- Cost as % of GDP ---
Mann-Kendall: trend=increasing, p=0.0204, Sen's slope=0.0063
Linear Regression: slope=0.0104, R²=0.1173, p=0.0213

--- Events per Million People ---
Mann-Kendall: trend=increasing, p=0.0000, Sen's slope=0.0011
Linear Regression: slope=0.0012, R²=0.6460, p=0.0000

--- Total Deaths ---
Mann-Kendall: trend=no trend, p=0.1648, Sen's slope=2.9773
Linear Regression: slope=7.3328, R²=0.0277, p=0.2747



In [9]:
# Acceleration analysis: is the rate of change itself changing?
# Approach: fit a quadratic (year + year²) and test if the year² term is significant.

from sklearn.linear_model import LinearRegression
from scipy.stats import f as f_dist
import numpy as np

def test_acceleration(series, label):
    """Test if a quadratic model fits significantly better than linear"""
    y = series.values
    x = analysis['year'].values
    x_centered = x - x.mean()  # center year to reduce collinearity

    # Linear model
    X_linear = x_centered.reshape(-1, 1)
    lin_model = LinearRegression().fit(X_linear, y)
    y_pred_linear = lin_model.predict(X_linear)
    rss_linear = np.sum((y - y_pred_linear)**2)

    # Quadratic model
    X_quad = np.column_stack([x_centered, x_centered**2])
    quad_model = LinearRegression().fit(X_quad, y)
    y_pred_quad = quad_model.predict(X_quad)
    rss_quad = np.sum((y - y_pred_quad)**2)

    # F-test: does the quadratic term significantly improve fit?
    n = len(y)
    df1 = 1
    df2 = n - 3
    f_stat = ((rss_linear - rss_quad) / df1) / (rss_quad / df2)
    p_value = 1 - f_dist.cdf(f_stat, df1, df2)

    year_squared_coef = quad_model.coef_[1]

    print(f"--- {label} ---")
    print(f"Linear R²:    {1 - rss_linear/np.sum((y - y.mean())**2):.4f}")
    print(f"Quadratic R²: {1 - rss_quad/np.sum((y - y.mean())**2):.4f}")
    print(f"Year² coefficient: {year_squared_coef:.6f}")
    print(f"F-test p-value for added curvature: {p_value:.4f}")
    print(f"Interpretation: {'Accelerating ↗↗' if year_squared_coef > 0 and p_value < 0.05 else 'Decelerating ↗↘' if year_squared_coef < 0 and p_value < 0.05 else 'No significant acceleration'}")
    print()

test_acceleration(analysis['event_count'], 'Event Count')
test_acceleration(analysis['total_cost_millions'], 'Total Cost (millions)')
test_acceleration(analysis['cost_pct_of_gdp'], 'Cost as % of GDP')
test_acceleration(analysis['events_per_million'], 'Events per Million')

--- Event Count ---
Linear R²:    0.7028
Quadratic R²: 0.8445
Year² coefficient: 0.016487
F-test p-value for added curvature: 0.0000
Interpretation: Accelerating ↗↗

--- Total Cost (millions) ---
Linear R²:    0.2842
Quadratic R²: 0.3159
Year² coefficient: 90.045088
F-test p-value for added curvature: 0.1708
Interpretation: No significant acceleration

--- Cost as % of GDP ---
Linear R²:    0.1173
Quadratic R²: 0.1334
Year² coefficient: 0.000333
F-test p-value for added curvature: 0.3818
Interpretation: No significant acceleration

--- Events per Million ---
Linear R²:    0.6460
Quadratic R²: 0.7834
Year² coefficient: 0.000046
F-test p-value for added curvature: 0.0000
Interpretation: Accelerating ↗↗



In [10]:
# Add a few rolling/cumulative metrics useful for visualization

# 5-year rolling averages — smooth out year-to-year noise for trend visualization
analysis['event_count_5yr_avg'] = analysis['event_count'].rolling(window=5, min_periods=1).mean()
analysis['cost_5yr_avg'] = analysis['total_cost_millions'].rolling(window=5, min_periods=1).mean()
analysis['cost_pct_gdp_5yr_avg'] = analysis['cost_pct_of_gdp'].rolling(window=5, min_periods=1).mean()

# Cumulative totals — useful for "total damage since 1980" visualizations
analysis['cumulative_events'] = analysis['event_count'].cumsum()
analysis['cumulative_cost_millions'] = analysis['total_cost_millions'].cumsum()
analysis['cumulative_deaths'] = analysis['total_deaths'].cumsum()

# Round numeric columns to reasonable precision for Tableau
for col in ['cost_pct_of_gdp', 'events_per_million', 'deaths_per_million',
            'event_count_5yr_avg', 'cost_5yr_avg', 'cost_pct_gdp_5yr_avg']:
    analysis[col] = analysis[col].round(4)

for col in ['cost_per_capita_dollars', 'avg_cost_per_event']:
    analysis[col] = analysis[col].round(2)

# Verify final shape
print(f"Final shape: {analysis.shape}")
print(f"Columns: {analysis.columns.tolist()}")
print()
print(analysis.head())

# Export to CSV
analysis.to_csv('climate_disasters_analysis.csv', index=False)
print("\n✓ Exported to climate_disasters_analysis.csv")

Final shape: (45, 19)
Columns: ['year', 'event_count', 'total_cost_millions', 'total_deaths', 'avg_cost_per_event', 'max_cost', 'gdp_billions', 'population', 'cost_pct_of_gdp', 'events_per_million', 'cost_per_capita_dollars', 'deaths_per_million', 'decade', 'event_count_5yr_avg', 'cost_5yr_avg', 'cost_pct_gdp_5yr_avg', 'cumulative_events', 'cumulative_cost_millions', 'cumulative_deaths']

   year  event_count  total_cost_millions  total_deaths  avg_cost_per_event  \
0  1980            3              45666.8          1280            15222.27   
1  1981            2               3485.5            20             1742.75   
2  1982            5              15175.7           243             3035.14   
3  1983            4              26755.5           186             6688.88   
4  1984            2               3208.0            81             1604.00   

   max_cost  gdp_billions  population  cost_pct_of_gdp  events_per_million  \
0   40681.2      7257.316   227225000           0.6293 

In [11]:
# Also export the event-level data for individual-disaster visualizations
disasters_export = disasters.copy()

# Add a year column already exists, but also add cost_per_event ranking and disaster type aggregates
disasters_export['cost_rank'] = disasters_export['cpi_adjusted_cost'].rank(ascending=False).astype(int)

# Round the costs to a sensible precision
disasters_export['cpi_adjusted_cost'] = disasters_export['cpi_adjusted_cost'].round(2)
disasters_export['unadjusted_cost'] = disasters_export['unadjusted_cost'].round(2)

# Export
disasters_export.to_csv('climate_disasters_events.csv', index=False)
print(f"Shape: {disasters_export.shape}")
print(f"Columns: {disasters_export.columns.tolist()}")
print()
print(disasters_export.head())
print("\n✓ Exported to climate_disasters_events.csv")

# Quick look at the top 10 most expensive events
print("\n--- Top 10 Most Expensive Events ---")
top10 = disasters_export.nsmallest(10, 'cost_rank')[['name', 'disaster_type', 'year', 'cpi_adjusted_cost', 'deaths']]
print(top10.to_string(index=False))

Shape: (403, 9)
Columns: ['name', 'disaster_type', 'begin_date', 'end_date', 'cpi_adjusted_cost', 'unadjusted_cost', 'deaths', 'year', 'cost_rank']

                                                name     disaster_type  \
0   Southern Severe Storms and Flooding (April 1980)          Flooding   
1                      Hurricane Allen (August 1980)  Tropical Cyclone   
2  Central/Eastern Drought/Heat Wave (Summer-Fall...           Drought   
3                      Florida Freeze (January 1981)            Freeze   
4  Severe Storms, Flash Floods, Hail, Tornadoes (...      Severe Storm   

  begin_date   end_date  cpi_adjusted_cost  unadjusted_cost  deaths  year  \
0 1980-04-10 1980-04-17             2749.4            706.8       7  1980   
1 1980-08-07 1980-08-11             2236.2            590.0      13  1980   
2 1980-06-01 1980-11-30            40681.2          10020.0    1260  1980   
3 1981-01-12 1981-01-14             2076.4            572.0       0  1981   
4 1981-05-05 1981-05-